<a href="https://colab.research.google.com/github/ayyanarh1/tamil-nadu-school-flood-risk/blob/main/github_com_ayyanarh1_tamil_nadu_school_flood_risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Cell 1 — Setup
!pip install earthengine-api geemap folium geopandas -q

import ee
ee.Authenticate()
import geemap
import pandas as pd
import folium

ee.Initialize(project='tamil-nadu-flood-risk')

print(ee.String('Day 2 ready!').getInfo())

Day 2 ready!


In [5]:
# Cell 3 — Load JRC Global Surface Water over Tamil Nadu
import ee
import geemap

ee.Initialize(project='tamil-nadu-flood-risk')

# Tamil Nadu boundary
tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

# JRC Global Surface Water — monthly history 1984 to 2021
# 'occurrence' band = % of months water was present
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
    .select('occurrence') \
    .clip(tamil_nadu)

# Display map
Map = geemap.Map()
Map.centerObject(tamil_nadu, zoom=7)

# JRC occurrence layer
# Blue = permanent water, light = occasional water, white = never flooded
Map.addLayer(
    jrc,
    {
        'min': 0, 'max': 100,
        'palette': ['ffffff', 'c0e4f7', '74bde0', '2a7ab5', '0d47a1']
    },
    'JRC Water Occurrence (1984-2021)'
)

print('JRC dataset loaded successfully!')
print('Blue = permanent water bodies')
print('Light blue = occasionally flooded areas')
print('White = never flooded')

Map

JRC dataset loaded successfully!
Blue = permanent water bodies
Light blue = occasionally flooded areas
White = never flooded


Map(center=[10.749705958455543, 78.24999999999999], controls=(WidgetControl(options=['position', 'transparent_…

In [8]:
# Cell 4 — Extract JRC water occurrence at each school location
import ee
import geemap
import pandas as pd

ee.Initialize(project='tamil-nadu-flood-risk')

# ── 1. School data ──
data = {
    'school_name': [
        'Govt High School Chennai', 'Govt School Cuddalore',
        'Panchayat School Nagapattinam', 'Govt School Thanjavur',
        'High School Coimbatore', 'Govt School Madurai',
        'Panchayat School Tirunelveli', 'Govt School Salem',
        'High School Vellore', 'Govt School Tiruchirappalli',
        'School Ramanathapuram', 'School Puducherry Border',
        'School Kanchipuram', 'School Villupuram',
        'School Tuticorin'
    ],
    'latitude': [
        13.08, 11.75, 10.76, 10.78, 11.01,
        9.93, 8.71, 11.65, 12.92, 10.79,
        9.37, 11.93, 12.83, 11.93, 8.80
    ],
    'longitude': [
        80.27, 79.75, 79.84, 79.13, 76.96,
        78.12, 77.69, 78.16, 79.13, 78.68,
        78.83, 79.83, 79.70, 79.49, 78.15
    ],
    'connectivity': [
        'connected', 'none', 'none',
        'connected', 'connected', 'connected',
        'none', 'connected', 'connected',
        'none', 'none', 'connected',
        'none', 'none', 'connected'
    ]
}

schools_df = pd.DataFrame(data)

# ── 2. Build GEE FeatureCollection with 1km buffer ──
features = []
for _, row in schools_df.iterrows():
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    buffer = point.buffer(1000)
    feature = ee.Feature(buffer, {
        'school_name': row['school_name'],
        'connectivity': row['connectivity'],
        'latitude': row['latitude'],
        'longitude': row['longitude']
    })
    features.append(feature)

schools_buffered = ee.FeatureCollection(features)

# ── 3. Load JRC occurrence ──
tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater') \
    .select('occurrence') \
    .clip(tamil_nadu)

# ── 4. Extract mean JRC occurrence per school buffer ──
jrc_stats = jrc.reduceRegions(
    collection=schools_buffered,
    reducer=ee.Reducer.mean(),
    scale=30  # JRC is 30m resolution
)

# ── 5. Parse results ──
jrc_list = jrc_stats.select(
    ['school_name', 'connectivity', 'mean']
).getInfo()

jrc_data = [f['properties'] for f in jrc_list['features']]
jrc_df = pd.DataFrame(jrc_data)

# Rename and clean
jrc_df = jrc_df.rename(columns={'mean': 'water_occurrence_pct'})
jrc_df['water_occurrence_pct'] = jrc_df['water_occurrence_pct'].fillna(0).round(2)

# Historical risk tier
def historical_risk(pct):
    if pct >= 20:
        return

In [9]:
# Cell 5 — Debug JRC extraction
print('Number of features returned:', len(jrc_list['features']))
print()
print('First feature properties:')
print(jrc_list['features'][0]['properties'])

Number of features returned: 15

First feature properties:
{'connectivity': 'connected', 'mean': 11.911558669001748, 'school_name': 'Govt High School Chennai'}


In [10]:
# Cell 6 — Fixed results display
import pandas as pd

# Parse results fresh
jrc_data = [f['properties'] for f in jrc_list['features']]
jrc_df = pd.DataFrame(jrc_data)

# Rename mean column
jrc_df = jrc_df.rename(columns={'mean': 'water_occurrence_pct'})
jrc_df['water_occurrence_pct'] = jrc_df['water_occurrence_pct'].fillna(0).round(2)

# Historical risk tier
def historical_risk(pct):
    if pct >= 20:
        return 'HIGH'
    elif pct >= 5:
        return 'MEDIUM'
    else:
        return 'LOW'

jrc_df['historical_risk'] = jrc_df['water_occurrence_pct'].apply(historical_risk)

# Sort by water occurrence
jrc_df = jrc_df.sort_values('water_occurrence_pct', ascending=False).reset_index(drop=True)

# Print results
print('=== JRC HISTORICAL WATER OCCURRENCE (1984-2021) ===\n')
for _, row in jrc_df.iterrows():
    print(f"{row['school_name'][:35]:<35} | "
          f"{row['connectivity']:<12} | "
          f"Occurrence: {row['water_occurrence_pct']:5.2f}% | "
          f"Risk: {row['historical_risk']}")

=== JRC HISTORICAL WATER OCCURRENCE (1984-2021) ===

School Puducherry Border            | connected    | Occurrence: 97.20% | Risk: HIGH
School Tuticorin                    | connected    | Occurrence: 60.95% | Risk: HIGH
Panchayat School Nagapattinam       | none         | Occurrence: 44.96% | Risk: HIGH
Govt School Thanjavur               | connected    | Occurrence: 43.46% | Risk: HIGH
School Ramanathapuram               | none         | Occurrence: 38.81% | Risk: HIGH
High School Vellore                 | connected    | Occurrence: 32.90% | Risk: HIGH
Govt School Cuddalore               | none         | Occurrence: 30.18% | Risk: HIGH
School Kanchipuram                  | none         | Occurrence: 29.21% | Risk: HIGH
Govt School Tiruchirappalli         | none         | Occurrence: 25.28% | Risk: HIGH
Panchayat School Tirunelveli        | none         | Occurrence: 19.47% | Risk: MEDIUM
Govt High School Chennai            | connected    | Occurrence: 11.91% | Risk: MEDIUM
Govt Sch

In [11]:
# Cell 7 — Combined risk score (SAR + JRC)
import pandas as pd

# Day 1 Sentinel-1 results
sar_data = {
    'school_name': [
        'School Puducherry Border', 'School Ramanathapuram',
        'Govt School Cuddalore', 'Panchayat School Nagapattinam',
        'Govt School Madurai', 'High School Vellore',
        'School Villupuram', 'School Tuticorin',
        'Govt School Tiruchirappalli', 'Govt School Thanjavur',
        'Govt High School Chennai', 'Panchayat School Tirunelveli',
        'High School Coimbatore', 'Govt School Salem',
        'School Kanchipuram'
    ],
    'connectivity': [
        'connected', 'none', 'none', 'none',
        'connected', 'connected', 'none', 'connected',
        'none', 'connected', 'connected', 'none',
        'connected', 'connected', 'none'
    ],
    'sar_flood_pct': [
        10.87, 1.59, 1.56, 1.07, 0.95,
        0.63, 0.63, 0.34, 0.34, 0.00,
        0.00, 0.00, 0.00, 0.00, 0.00
    ]
}
sar_df = pd.DataFrame(sar_data)

# Merge SAR + JRC
combined = sar_df.merge(
    jrc_df[['school_name', 'water_occurrence_pct', 'historical_risk']],
    on='school_name'
)

# ── Combined risk score ──
# Normalize SAR (0-100 scale) — max observed was 10.87%
combined['sar_norm'] = (combined['sar_flood_pct'] / 10.87 * 100).round(2)

# JRC already 0-100 scale
combined['jrc_norm'] = combined['water_occurrence_pct'].round(2)

# Combined score — 40% SAR (recent event) + 60% JRC (historical pattern)
combined['combined_score'] = (
    0.4 * combined['sar_norm'] +
    0.6 * combined['jrc_norm']
).round(2)

# Connectivity penalty — no connectivity adds 10 points to score
combined['connectivity_penalty'] = combined['connectivity'].apply(
    lambda x: 10 if x == 'none' else 0
)
combined['final_score'] = (
    combined['combined_score'] + combined['connectivity_penalty']
).round(2)

# Final risk tier
def final_risk(score):
    if score >= 40:
        return 'CRITICAL'
    elif score >= 20:
        return 'HIGH'
    elif score >= 10:
        return 'MEDIUM'
    else:
        return 'LOW'

combined['final_risk'] = combined['final_score'].apply(final_risk)

# Sort by final score
combined = combined.sort_values('final_score', ascending=False).reset_index(drop=True)
combined['rank'] = combined.index + 1

# Print final table
print('=== COMBINED SCHOOL FLOOD RISK RANKING ===')
print('(SAR 2023 + JRC Historical + Connectivity penalty)\n')
for _, row in combined.iterrows():
    conn_flag = '⚠' if row['connectivity'] == 'none' else ' '
    print(f"#{int(row['rank']):<2} {conn_flag} {row['school_name'][:32]:<32} | "
          f"Score: {row['final_score']:5.1f} | "
          f"Risk: {row['final_risk']}")

=== COMBINED SCHOOL FLOOD RISK RANKING ===
(SAR 2023 + JRC Historical + Connectivity penalty)

#1    School Puducherry Border         | Score:  98.3 | Risk: CRITICAL
#2  ⚠ Panchayat School Nagapattinam    | Score:  40.9 | Risk: CRITICAL
#3  ⚠ School Ramanathapuram            | Score:  39.1 | Risk: HIGH
#4    School Tuticorin                 | Score:  37.8 | Risk: HIGH
#5  ⚠ Govt School Cuddalore            | Score:  33.9 | Risk: HIGH
#6  ⚠ School Kanchipuram               | Score:  27.5 | Risk: HIGH
#7  ⚠ Govt School Tiruchirappalli      | Score:  26.4 | Risk: HIGH
#8    Govt School Thanjavur            | Score:  26.1 | Risk: HIGH
#9    High School Vellore              | Score:  22.1 | Risk: HIGH
#10 ⚠ Panchayat School Tirunelveli     | Score:  21.7 | Risk: HIGH
#11 ⚠ School Villupuram                | Score:  14.0 | Risk: MEDIUM
#12   Govt School Madurai              | Score:   9.4 | Risk: LOW
#13   Govt High School Chennai         | Score:   7.2 | Risk: LOW
#14   High School Coimbato

In [12]:
# Cell 8 — Final combined risk map
import folium
import pandas as pd

# Add coordinates back to combined dataframe
coords = {
    'School Puducherry Border':       [11.93, 79.83],
    'Panchayat School Nagapattinam':  [10.76, 79.84],
    'School Ramanathapuram':          [9.37,  78.83],
    'School Tuticorin':               [8.80,  78.15],
    'Govt School Cuddalore':          [11.75, 79.75],
    'School Kanchipuram':             [12.83, 79.70],
    'Govt School Tiruchirappalli':    [10.79, 78.68],
    'Govt School Thanjavur':          [10.78, 79.13],
    'High School Vellore':            [12.92, 79.13],
    'Panchayat School Tirunelveli':   [8.71,  77.69],
    'School Villupuram':              [11.93, 79.49],
    'Govt School Madurai':            [9.93,  78.12],
    'Govt High School Chennai':       [13.08, 80.27],
    'High School Coimbatore':         [11.01, 76.96],
    'Govt School Salem':              [11.65, 78.16],
}

combined['latitude']  = combined['school_name'].map(lambda x: coords[x][0])
combined['longitude'] = combined['school_name'].map(lambda x: coords[x][1])

# ── Build map ──
m = folium.Map(location=[10.5, 78.5], zoom_start=7, tiles='CartoDB positron')

# Colour by final risk tier
def get_color(risk):
    return {
        'CRITICAL': '#cc0000',
        'HIGH':     '#ff6600',
        'MEDIUM':   '#ffaa00',
        'LOW':      '#2d8a4e'
    }.get(risk, 'gray')

# Radius by score
def get_radius(score):
    if score >= 40:
        return 18
    elif score >= 20:
        return 14
    elif score >= 10:
        return 10
    else:
        return 7

# Add each school
for _, row in combined.iterrows():
    color  = get_color(row['final_risk'])
    radius = get_radius(row['final_score'])
    conn_flag = '⚠ No connectivity' if row['connectivity'] == 'none' else '✅ Connected'

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        tooltip=f"#{int(row['rank'])} {row['school_name']}",
        popup=folium.Popup(
            f"""
            <b>#{int(row['rank'])} {row['school_name']}</b><br>
            <hr style='margin:4px 0'>
            🌊 <b>SAR flood exposure:</b> {row['sar_flood_pct']}%<br>
            📅 <b>Historical occurrence:</b> {row['water_occurrence_pct']}%<br>
            📊 <b>Combined score:</b> {row['final_score']}<br>
            📡 <b>Connectivity:</b> {conn_flag}<br>
            <hr style='margin:4px 0'>
            🎯 <b>Risk tier: {row['final_risk']}</b>
            """,
            max_width=280
        )
    ).add_to(m)

    # Add rank label
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:white;'
                 f'text-align:center;margin-top:3px">#{int(row["rank"])}</div>',
            icon_size=(20, 20),
            icon_anchor=(10, 10)
        )
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;
     background:white;padding:14px;border-radius:10px;
     border:1px solid #ccc;font-size:12px;z-index:1000;
     box-shadow:2px 2px 6px rgba(0,0,0,0.2)">
     <b>🏫 School Flood Risk</b><br>
     <b>Combined SAR + JRC Score</b><br><br>
     <span style='color:#cc0000;font-size:16px'>●</span> Critical (score ≥ 40)<br>
     <span style='color:#ff6600;font-size:16px'>●</span> High (score 20–40)<br>
     <span style='color:#ffaa00;font-size:16px'>●</span> Medium (score 10–20)<br>
     <span style='color:#2d8a4e;font-size:16px'>●</span> Low (score < 10)<br><br>
     <i>⚠ = no internet connectivity</i><br>
     <i>Size = risk score magnitude</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Save
m.save('tamil_nadu_combined_risk_map.html')
print('✅ Combined risk map saved!')
m

✅ Combined risk map saved!


In [13]:
# Cell 9 — Download the map
from google.colab import files
files.download('tamil_nadu_combined_risk_map.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# Cell 10 — Save risk table as CSV
combined.to_csv('tamil_nadu_combined_risk_scores.csv', index=False)
files.download('tamil_nadu_combined_risk_scores.csv')
print('✅ CSV downloaded!')
print(f'Total schools analysed: {len(combined)}')
print(f'CRITICAL risk schools: {len(combined[combined.final_risk == "CRITICAL"])}')
print(f'HIGH risk schools: {len(combined[combined.final_risk == "HIGH"])}')
print(f'Schools with no connectivity in top 10: {len(combined.head(10)[combined.head(10).connectivity == "none"])}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ CSV downloaded!
Total schools analysed: 15
CRITICAL risk schools: 2
HIGH risk schools: 8
Schools with no connectivity in top 10: 6
